# kumo-relational on a local NIM

Several related tables, joined as a graph, and a question asked in PQL about
what an entity reaches in a window.

This needs `pip install 'structured-data-models-client[relational]'` and a
`kumo-relational` NIM. One table on its own is a separate notebook,
[`tabular.ipynb`](tabular.ipynb).

In [1]:
import os

import pandas as pd
from sdm_client import KumoRelational, PredictiveQuery, Schema, SDMClient

# Point this at your own deployment, or set the variable instead.
RELATIONAL_URL = os.environ.get('SDM_RELATIONAL_URL', 'http://localhost:8001')

In [2]:
n_users = 40
users = pd.DataFrame({'user_id': range(n_users)})

rows, order_id = [], 0
for user in range(n_users):
    busy = user % 2 == 0
    for k in range(40 if busy else 4):
        rows.append(
            {
                'order_id': order_id,
                'user_id': user,
                'price': 10.0 + (user % 7) * 3 + k,
                # A 30-day aggregation needs 30 days of history before the anchor.
                'ts': pd.Timestamp('2024-01-01')
                + pd.Timedelta(days=(k * 5 if busy else k * 50) + user % 9),
            }
        )
        order_id += 1
orders = pd.DataFrame(rows)

schema = Schema.from_data({'users': users, 'orders': orders})
schema

Schema(2 tables: ['orders', 'users'], 1 relationships)

Everything is inferred: dtypes, semantic types, keys, time columns and the
relationship between the two tables. Override only where inference is wrong.

A code stored as a number reads as a number, so a postcode or a status code
needs `schema["t"]["col"].stype = "categorical"`.


In [3]:
schema['orders'], schema['users'].primary_key, schema.relationships

(Table('orders', 880 rows, 4 columns, primary_key='order_id', time_column='ts'),
 'user_id',
 [orders.user_id -> users.user_id])

In [4]:
query = PredictiveQuery(
    'PREDICT COUNT(orders.*, 0, 30, days) FOR EACH users.user_id',
    schema,
)

# The gathering step is separate because it is the expensive one: it resolves
# the query, finds labelled examples in history, samples each entity's
# neighbourhood and encodes the result. Nothing is sent here.
relational_request = query.materialize(indices=[0, 1, 2, 3])
relational_request

Request(regression, target='TARGET', 240 context rows, 4 to score, relational)

In [5]:
with SDMClient(url=RELATIONAL_URL, timeout=600.0) as client:
    relational_result = client.predict(
        relational_request, model=KumoRelational()
    )

relational_result.to_pandas()

,row_index,id,prediction
0,0,0,5.828968
1,1,1,0.020786
2,2,2,5.828968
3,3,3,0.020786


Even-numbered users order roughly ten times as often as odd ones, and the
predictions should say so.

## Asking one context several questions

Uploading a context once and scoring against it repeatedly is what `fit` is
for. The pinned half is model, task, schema and context; later calls send only
the rows.


In [6]:
with SDMClient(url=RELATIONAL_URL, timeout=600.0) as client:
    with client.fit(relational_request, model=KumoRelational()) as fitted:
        print('session:', fitted.session_id)
        display(fitted.predict(relational_request).to_pandas())

session: sess_d4dda93c33cb


,row_index,id,prediction
0,0,0,5.828968
1,1,1,0.020786
2,2,2,5.828968
3,3,3,0.020786
